# Paper 1 Exploration: Agricultural Transition Pathways


## Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Add project root to path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

from analysis.paper1_escape.trajectories import extract_trajectory_features
from analysis.paper1_escape.clustering import (
    prepare_clustering_features, cluster_pathways, label_clusters,
)
from analysis.paper1_escape.survival import build_survival_dataset, fit_cox_model

PANEL_PATH = os.path.join(os.path.dirname(os.getcwd()), 'data', 'country_analysis_panel.parquet')
panel = pd.read_parquet(PANEL_PATH)
print(f'Panel shape: {panel.shape}')
print(panel.dtypes)
panel.head()


## Trajectory Extraction


In [ ]:
traj_df = extract_trajectory_features(panel, entity_col='country')
print(f'Trajectory features shape: {traj_df.shape}')
print(f'Countries with all features: {traj_df.dropna().shape[0]}')
traj_df.describe()


## Clustering Scan (k=2..6)


In [ ]:
X_scaled, entities = prepare_clustering_features(traj_df, entity_col='country')

scores = {}
for k in range(2, 7):
    labels, metrics = cluster_pathways(X_scaled, n_clusters=k)
    scores[k] = metrics['silhouette']
    print(f'k={k}: silhouette={metrics["silhouette"]:.4f}, inertia={metrics["inertia"]:.2f}')

best_k = max(scores, key=scores.get)
print(f'\nBest k by silhouette score: {best_k}')

plt.figure(figsize=(8, 4))
plt.plot(list(scores.keys()), list(scores.values()), marker='o')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Silhouette score')
plt.title('Clustering scan: silhouette scores')
plt.axvline(best_k, color='red', linestyle='--', label=f'Best k={best_k}')
plt.legend()
plt.tight_layout()
plt.show()


## Final Clustering


In [ ]:
final_labels, final_metrics = cluster_pathways(X_scaled, n_clusters=best_k)
cluster_df = label_clusters(entities, final_labels, entity_col='country')

# Merge cluster labels back onto trajectory features
traj_with_clusters = traj_df.merge(cluster_df, on='country', how='inner')

print(f'Final clustering (k={best_k}): silhouette={final_metrics["silhouette"]:.4f}')
print('\nCluster sizes:')
print(traj_with_clusters['cluster'].value_counts().sort_index())

print('\nCluster summary (feature means):')
feature_cols = [
    'peak_extensification_year', 'intensification_onset_year',
    'urbanization_takeoff_year', 'max_pop_density', 'min_land_labor_ratio',
]
print(traj_with_clusters.groupby('cluster')[feature_cols].mean().round(2))


## Survival Analysis


In [ ]:
surv_df = build_survival_dataset(
    traj_df,
    event_col='intensification_onset_year',
    origin_col='peak_extensification_year',
    entity_col='country',
)
print(f'Survival dataset shape: {surv_df.shape}')
print(f'Events observed: {surv_df["event"].sum()} / {len(surv_df)}')
print(surv_df.describe())

# Merge in covariates from trajectory features
covariate_cols = ['max_pop_density', 'min_land_labor_ratio']
surv_full = surv_df.merge(
    traj_df[['country'] + covariate_cols].dropna(),
    on='country', how='inner',
)
print(f'\nSurvival dataset with covariates: {surv_full.shape}')

if len(surv_full) >= 10 and surv_full['event'].sum() >= 3:
    cox_summary = fit_cox_model(
        surv_full,
        covariates=covariate_cols,
    )
    print('\nCox PH model summary:')
    print(cox_summary)
else:
    print('Insufficient events for Cox model fitting.')
